# Module 9: RAG & LLM Applications
## LLM Optimization & Evaluation: 05 - Semantic & Exact Prompt Caching

### Theoretical Overview
In enterprise apps, up to $40\%$ of user queries are semantically repetitive.
- **Exact Caching**: SHA-256 hash match on exact prompt text (zero tolerance for typos).
- **Semantic Caching**: Embedding vector search in Redis/GPTCache. If cosine similarity $> 0.95$, return cached response instantly with $0$ LLM cost.

In [ ]:
import numpy as np

class SemanticCache:
    def __init__(self, threshold: float = 0.92):
        self.threshold = threshold
        self.cache = [] # (vector, response)
        
    def get(self, query_vector: np.ndarray):
        q_norm = query_vector / np.linalg.norm(query_vector)
        for vec, resp in self.cache:
            v_norm = vec / np.linalg.norm(vec)
            sim = float(np.dot(q_norm, v_norm))
            if sim >= self.threshold:
                return resp, sim
        return None, 0.0
        
    def set(self, query_vector: np.ndarray, response: str):
        self.cache.append((query_vector, response))

cache = SemanticCache(threshold=0.90)
v_orig = np.array([0.9, 0.1, 0.1])
cache.set(v_orig, "Python was released in 1991 by Guido van Rossum.")

# Slight variation query vector
v_variation = np.array([0.89, 0.11, 0.12])
hit, sim = cache.get(v_variation)
print(f"Cache Hit: {hit is not None} | Cosine Sim: {sim:.4f}")
print(f"Cached Response: {hit}")